# Plotting Climate Anomalies as a Function of Two Teleconnection Indices
## By Landon Moeller

### Importing Packages

In [1]:
import xarray as xr
import pandas as pd
import numpy as np
import math
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

### Opening Data

In [2]:
nh_ds = xr.open_dataset("../Data/NH_tele_cli_anoms_detrended_1950-2024.nc")
sh_ds = xr.open_dataset("../Data/SH_tele_cli_anoms_detrended_1950-2024.nc")
global_ds = xr.open_dataset("../Data/Global_tele_cli_anoms_detrended_1950-2024.nc")
ds = xr.merge([nh_ds, sh_ds, global_ds], compat='override').compute()
ds

<xarray.Dataset> Size: 591kB
Dimensions:                                  (year: 75, month: 12)
Coordinates:
  * year                                     (year) int64 600B 1950 ... 2024
  * month                                    (month) int64 96B 1 2 3 ... 11 12
    time                                     (year, month) datetime64[ns] 7kB ...
Data variables: (12/81)
    TNA                                      (year, month) float64 7kB -0.29 ...
    WPO                                      (year, month) float64 7kB -2.64 ...
    WP                                       (year, month) float64 7kB -0.95 ...
    SCA                                      (year, month) float64 7kB 0.78 ....
    EAWR                                     (year, month) float64 7kB 1.95 ....
    EA                                       (year, month) float64 7kB -2.62 ...
    ...                                       ...
    IOD                                      (year, month) float64 7kB 0.45 ....
    EMI                                      (year, month) float64 7kB -0.43 ...
    QBO                                      (year, month) float64 7kB -9.0 ....
    MEI                                      (year, month) float64 7kB -1.34 ...
    AAM                                      (year, month) float64 7kB nan .....
    ENSO_34                                  (year, month) float64 7kB -1.98 ...
Attributes:
    Name:                   NH Teleconnections and Regional Climate Anomalies...
    Teleconnection_source:  Climate Prediction Center
    Anomaly_source:         ECMWF Reanalysis v5 (ERA5)

# Plotting

Running this widget allows the user to choose two teleconnection indices, the month, variable, and region.

In [4]:
teleconnections = ['TNA', 'WPO', 'WP', 'SCA', 'EAWR', 'EA', 'ADI', 'AMO',
                   'AO', 'EPNP', 'EPO', 'NAO', 'NOI', 'PDO', 'PNA', 'POL',
                   'SAOD', 'TSA', 'SPOD', 'TPI', 'SOI', 'AAO', 'IPO', 'IOD',
                   'EMI', 'QBO', 'MEI', 'AAM', 'ENSO_34']

month_names = {1: "January", 2: "February", 3: "March", 4: "April", 5: "May", 6: "June",
               7: "July", 8: "August", 9: "September", 10: "October", 11: "November", 12: "December"}

regions = sorted(list(set([var.replace("T2M_StdAnom_", "").replace("TP_StdAnom_", "").replace("_Detrended", "")
                           for var in ds.data_vars if "StdAnom" in var])))

variable_options = {"Std Temp Anomaly": "T2M_StdAnom", "Std Precip Anomaly": "TP_StdAnom"}

# Plotting function
def plot_teleconnection_combination(ds, tele_x, tele_y, anomaly_var, month=None):

    if "TP_" in anomaly_var:
        cmap = "BrBG"
        plot_label = "Std Precip Anomaly"
    else:
        cmap = "seismic"
        plot_label = "Std Temp Anomaly"

    if month is not None:
        ds_plot = ds.sel(month=month)
        month_title = month_names[month]
        month_save = month_names[month].replace(" ", "_")
    else:
        ds_plot = ds.stack(sample=("year", "month"))
        month_title = "All Months"
        month_save = "AllMonths"

    x = ds_plot[tele_x].values.ravel()
    y = ds_plot[tele_y].values.ravel()
    c = ds_plot[anomaly_var].values.ravel()

    valid = ~np.isnan(x) & ~np.isnan(y) & ~np.isnan(c)

    x = x[valid]
    y = y[valid]
    c = c[valid]

    if len(x) == 0:
        return

    xlim_abs = math.ceil(np.nanmax(np.abs(x)))
    ylim_abs = math.ceil(np.nanmax(np.abs(y)))
    clim_abs = np.nanmax(np.abs(c))

    region = (anomaly_var.replace("T2M_StdAnom_", "").replace("TP_StdAnom_", "").replace("_Detrended", "").replace("_", " "))

    fig, ax = plt.subplots(figsize=(9, 6), dpi=400)

    sc = ax.scatter(x, y, c=c, cmap=cmap, marker='x', s=60, alpha=0.8, vmin=-clim_abs, vmax=clim_abs, zorder=2)

    ax.axhline(0, color='gray', lw=0.9, alpha=0.7, zorder=0)
    ax.axvline(0, color='gray', lw=0.9, alpha=0.7, zorder=0)

    ax.set_xlim((-xlim_abs, xlim_abs))
    ax.set_ylim((-ylim_abs, ylim_abs))

    ax.grid(True, linestyle='--', alpha=0.3)

    ax.set_xlabel(tele_x, fontsize=11, fontweight='bold')
    ax.set_ylabel(tele_y, fontsize=11, fontweight='bold')

    ax.set_title(f"{tele_x} vs {tele_y} — {plot_label} ({region})", fontsize=13, fontweight='bold')

    plt.figtext(0.761, 0.8425, f'{month_title}',
                ha='right', color='dimgray', fontsize=12, fontweight='bold',
                bbox=dict(facecolor='white', alpha=0.5, edgecolor='lightgray'))

    cbar = plt.colorbar(sc, ax=ax, pad=0.015, aspect=30)
    cbar.set_label(plot_label, fontsize=10)

    save_name = (f"{region}_{plot_label}_{tele_x}_vs_{tele_y}_{month_save}.png".replace(" ", "_"))
    plt.savefig(save_name, dpi=400, bbox_inches='tight')

    plt.show()

# Interactive widget
def interactive_teleconnection_plot(ds):

    tele1_dropdown = widgets.Dropdown(
        options=teleconnections,
        value='PNA',
        description='Index 1:',
        layout=widgets.Layout(width='300px')
    )

    tele2_dropdown = widgets.Dropdown(
        options=teleconnections,
        value='AO',
        description='Index 2:',
        layout=widgets.Layout(width='300px')
    )

    month_dropdown = widgets.Dropdown(
        options=[("All Months", None)] +
                [(month_names[m], m) for m in range(1, 13)],
        value=None,
        description='Month:'
    )

    variable_dropdown = widgets.Dropdown(
        options=list(variable_options.keys()),
        value='Std Temp Anomaly',
        description='Variable:',
        layout=widgets.Layout(width='250px')
    )

    region_dropdown = widgets.Dropdown(
        options=regions,
        value=regions[0],
        description='Region:',
        layout=widgets.Layout(width='300px')
    )

    button = widgets.Button(
        description="Build Plot",
        button_style='success'
    )

    output = widgets.Output()

    def on_button_clicked(b):
        with output:
            output.clear_output(wait=True)

            anomaly_var = (f"{variable_options[variable_dropdown.value]}_{region_dropdown.value}_Detrended")

            if anomaly_var not in ds.data_vars:
                return

            plot_teleconnection_combination(
                ds=ds,
                tele_x=tele1_dropdown.value,
                tele_y=tele2_dropdown.value,
                anomaly_var=anomaly_var,
                month=month_dropdown.value
            )

    button.on_click(on_button_clicked)

    display(widgets.VBox([
        tele1_dropdown,
        tele2_dropdown,
        month_dropdown,
        variable_dropdown,
        region_dropdown,
        button,
        output
    ]))

interactive_teleconnection_plot(ds)